In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import uuid, datetime, hashlib, json
# 2. Define make_event(event_type, payload, source="system") -> dict:
#    - "id":              str(uuid.uuid4())[:8]
#    - "type":            event_type
#    - "source":          source
#    - "payload":         payload
#    - "timestamp":       datetime.utcnow().isoformat() + "Z"
#    - "idempotency_key": sha256(event_type + json.dumps(payload, sort_keys=True))[:16]
# 3. Create 3 events: ORDER_PLACED, INVENTORY_CHECKED, SUPPLIER_NOTIFIED
# 4. Print each event's type, id, and idempotency_key
#
# Hint:
#   def make_event(event_type, payload, source="system"):
#       key_src = event_type + json.dumps(payload, sort_keys=True)
#       return {
#           "id":              str(uuid.uuid4())[:8],
#           "type":            event_type,
#           "source":          source,
#           "payload":         payload,
#           "timestamp":       datetime.datetime.utcnow().isoformat() + "Z",
#           "idempotency_key": hashlib.sha256(key_src.encode()).hexdigest()[:16],
#       }

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define class SimpleMessageBus:
#    a. __init__(self): self.queues = {}, self.handlers = {}
#    b. subscribe(self, topic, handler): self.handlers.setdefault(topic, []).append(handler)
#    c. publish(self, event): self.queues.setdefault(event["type"], []).append(event)
#    d. process(self, topic) -> list: pop all events from queue, call each handler,
#       return [(handler_name, event_id), ...]
# 2. Create bus = SimpleMessageBus()
# 3. Register 2 handlers for ORDER_PLACED; publish 2 events
# 4. Call bus.process("ORDER_PLACED") and print dispatch results
#
# Hint:
#   class SimpleMessageBus:
#       def __init__(self): self.queues = {}; self.handlers = {}
#       def subscribe(self, topic, handler):
#           self.handlers.setdefault(topic, []).append(handler)
#       def publish(self, event):
#           self.queues.setdefault(event["type"], []).append(event)
#       def process(self, topic):
#           events = self.queues.pop(topic, [])
#           return [(h.__name__, e["id"]) for e in events
#                   for h in self.handlers.get(topic, [])]

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Fan-out: publish 1 ORDER_PLACED event to 3 independent consumers
#    (inventory_consumer, pricing_consumer, audit_consumer)
# 2. Each consumer returns {"consumer": name, "status": "ok", "event_id": ...}
# 3. Fan-in: collect all 3 results
# 4. Print fan-out dispatch and fan-in results
# 5. Mark "ready_for_approval" only if all 3 consumers returned "ok"
#
# Hint:
#   consumers = [inventory_consumer, pricing_consumer, audit_consumer]
#   results   = [c(event) for c in consumers]   # fan-out + fan-in
#   all_ok    = all(r["status"] == "ok" for r in results)
#   print("Fan-out:", [f"{c.__name__} <- {event['id']}" for c in consumers])
#   print("Fan-in:", results)
#   print("Ready for approval" if all_ok else "Blocked")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define class DeadLetterQueue with add(event, reason) and messages list
# 2. Add DLQ to SimpleMessageBus; define flaky_handler(event) that raises
#    RuntimeError on first MAX_RETRIES-1 attempts then succeeds
# 3. Define process_with_retry(bus, topic, max_retries=3):
#    Try each handler up to max_retries times; on final failure: bus.dlq.add(event, reason)
# 4. Publish 3 events; call process_with_retry(); print DLQ size
#
# Hint:
#   MAX_RETRIES = 3; attempt_counts = {}
#   def flaky_handler(event):
#       eid = event["id"]
#       attempt_counts[eid] = attempt_counts.get(eid, 0) + 1
#       if attempt_counts[eid] < MAX_RETRIES:
#           raise RuntimeError("Transient error")
#   # After max_retries exhausted:
#   bus.dlq.add(event, reason=str(last_exception))
#   print("DLQ size:", len(bus.dlq.messages))

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define class IdempotentBus(SimpleMessageBus):
#    a. __init__: super().__init__(); self.processed_keys = set()
#    b. publish(event) -> bool:
#       key = event["idempotency_key"]
#       If key in processed_keys: print skip message; return False
#       Call super().publish(event); add key; return True
# 2. Create ibus = IdempotentBus(); register a handler
# 3. Publish the SAME event twice (same payload = same key)
# 4. Publish a DIFFERENT event
# 5. Print total attempts and how many were deduplicated
#
# Hint:
#   class IdempotentBus(SimpleMessageBus):
#       def __init__(self): super().__init__(); self.processed_keys = set()
#       def publish(self, event):
#           key = event["idempotency_key"]
#           if key in self.processed_keys:
#               print(f"  [SKIP] duplicate key: {key}"); return False
#           super().publish(event); self.processed_keys.add(key); return True

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define class CorrelationTracer:
#    a. __init__: self.traces = {}
#    b. record(event): cid = event.get("correlation_id", event["id"])
#       self.traces.setdefault(cid, []).append(event["id"])
#    c. get_trace(correlation_id) -> list
#    d. summary() -> dict: {cid: len(events)}
# 2. Create tracer; simulate 4-step order flow sharing one correlation_id:
#    ORDER_PLACED -> INVENTORY_CHECKED -> SUPPLIER_NOTIFIED -> ORDER_CONFIRMED
# 3. Call tracer.record(event) for each; print get_trace and summary
#
# Hint:
#   class CorrelationTracer:
#       def __init__(self): self.traces = {}
#       def record(self, event):
#           cid = event.get("correlation_id", event["id"])
#           self.traces.setdefault(cid, []).append(event["id"])
#   cid = "order-" + str(uuid.uuid4())[:6]
#   for etype in ["ORDER_PLACED","INVENTORY_CHECKED","SUPPLIER_NOTIFIED","ORDER_CONFIRMED"]:
#       e = make_event(etype, {"order_id": "ORD-1"})
#       e["correlation_id"] = cid; tracer.record(e)